In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

# Load Datasets

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

## Extract TF-IDF + Cosine Similarity Features

In [4]:
def extract_features(df, tfidf_vectorizer=None, is_train=True):
    # Combine all text fields to fit the TF-IDF space properly
    all_texts = df['prompt'].fillna("").tolist()
    for col in ['A', 'B', 'C', 'D', 'E']:
        all_texts += df[col].fillna("").tolist()
        
    if is_train:
        tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
        tfidf_vectorizer.fit(all_texts)
        
    features_list = []
    
    # Process row by row to generate cosine features
    for idx, row in df.iterrows():
        p_text = str(row['prompt'])
        options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
        
        # Vectorize prompt and options
        p_vec = tfidf_vectorizer.transform([p_text])
        opt_vecs = tfidf_vectorizer.transform(options)
        
        # Calculate cosine similarities
        sims = cosine_similarity(p_vec, opt_vecs).flatten() # Array of 5 similarity values
        
        # Add a few extra structural features per option (e.g., character length ratio)
        p_len = max(len(p_text), 1)
        len_ratios = [len(opt) / p_len for opt in options]
        
        # Total feature vector for this MCQ instance: 5 similarities + 5 length ratios = 10 features
        row_features = np.concatenate([sims, len_ratios])
        features_list.append(row_features)
        
    return np.array(features_list), tfidf_vectorizer

In [5]:
X_train, vectorizer = extract_features(train_df, is_train=True)
X_test, _ = extract_features(test_df, tfidf_vectorizer=vectorizer, is_train=False)

In [6]:
# Encode targets (A->0, B->1, etc.)
le = LabelEncoder()
y_train = le.fit_transform(train_df['answer'])

In [7]:
# Convert data to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

In [8]:
# 3. Define the Forward Neural Network Architecture
class MCQForwardNN(nn.Module):
    def __init__(self, input_dim):
        super(MCQForwardNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 5) # 5 Output logits corresponding to classes A, B, C, D, E
        )
        
    def forward(self, x):
        return self.network(x)

In [9]:
# Initialize Model, Loss, and Optimizer
model = MCQForwardNN(input_dim=X_train.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=0.01)

In [10]:
# Create DataLoader
dataset = TensorDataset(X_train_t, y_train_t)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [11]:
# 4. Model Training Loop
model.train()
for epoch in range(40):
    total_loss = 0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

In [12]:
# 5. Prediction and Top 3 MAP Ranking
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    # Apply softmax to get clear probability spreads
    probs = torch.softmax(test_outputs, dim=1).numpy()

In [13]:
# Convert probabilities to ranked character predictions (Top 3)
classes = np.array(['A', 'B', 'C', 'D', 'E'])
predictions = []

for row in probs:
    # Sort option indices from highest probability to lowest
    top_3_indices = np.argsort(row)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    predictions.append(" ".join(top_3_labels))

In [15]:
# 6. Generate Submission File
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions
})
submission.to_csv('submission.csv', index=False)
submission.head()
print("Submission file successfully generated and saved!")
submission.head()



Submission file successfully generated and saved!


,id,prediction
0,1,D B A
1,2,B C D
2,3,B E C
3,4,E D B
4,5,B E C
